# Temporal-consistency distillation with the final ConvNeXt (fold 0)

Repeats the earlier ResNet experiment on the final architecture. Everything except the networks is identical to it:
the same GONG bursts (2023+, never overlapping MAGFiLO's 2011–2022), the same clean-frame flags, alignment shifts,
student frames and held-out split.

* **Teacher:** the final ConvNeXt fold-0 model (it never saw the fold-0 validation images).
* **Students** (ConvNeXt, ImageNet start, the final training recipe on folds 1–3, plus two frames of one burst per
  step from epoch 2, full weight from epoch 7):
  * `demet_tc` — burst target (mean teacher prediction over the burst's clean frames) + frame-to-frame consistency;
  * `tekkare` — each frame's own teacher prediction (classic self-training; same unlabeled frames, no temporal
    information) — the control that separates "temporal information" from "more images".
* **Baseline:** the final fold-0 model itself.
* **Accuracy:** fold-0 PQ (scorer cross-fitted inside fold 0). **Stability:** self-PQ on the 80 held-out bursts.

In [ ]:
import os, sys, json, time, glob
from pathlib import Path
REPO_DIR = Path(globals().get("REPO_DIR", ".."))
DATA_DIR = Path(globals().get("DATA_DIR", "/kaggle/input"))
GONG_DIR = Path(globals().get("GONG_DIR", "gong_f0"))          # kareler.csv, demetler.csv, ayrik/
RAW_DIR = Path(globals().get("RAW_DIR", "gong_raw"))           # raw GONG frames (YYYYMMDDHHMMSS<station>h.jpg)
TEACHER = Path(globals().get("TEACHER", "weights/convnext_fold0.pt"))
WORK_DIR = Path(globals().get("WORK_DIR", "./work_temporal"))  # checkpoints, states, results
TARGET_DIR = Path(globals().get("TARGET_DIR", WORK_DIR / "targets"))
CACHE_DIR = Path(globals().get("CACHE_DIR", WORK_DIR / "cache"))
TIME_BUDGET_H = globals().get("TIME_BUDGET_H", None)
WORKERS = int(globals().get("WORKERS", max(1, min(8, os.cpu_count() or 2))))
DRY = os.environ.get("DRY") == "1"
FOLD = 0
VARIANTS = {"demet_tc": dict(target="burst", consistency=True), "tekkare": dict(target="single", consistency=False)}
RUN = globals().get("RUN", list(VARIANTS))

sys.path.insert(0, str(REPO_DIR))
import numpy as np, pandas as pd, cv2, torch
from filament.config import load_config, scaled
from filament.data import Annotations, make_folds, build_cache, find_files, read_cache
from filament.model import load_model
from filament.train import train_model
from filament import pipeline as pl, evaluate as ev, temporal as tp

cfg = load_config(REPO_DIR / "configs" / "final.yaml")
if DRY:
    cfg = scaled(cfg, 256)
    cfg["train"].update(epochs=3, val_every=1, workers=0)
WORK_DIR.mkdir(parents=True, exist_ok=True)
log = pl.log_to(WORK_DIR / "temporal.log")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = cfg["seed"]
log(f"device {device} ({torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'}) | variants {RUN}")

In [ ]:
# ------------------------------------------------------------------------------------- labeled data, fold 0
ann = Annotations(find_files(DATA_DIR, "*kaggle2026_train.json")[0],
                  [d for d in find_files(DATA_DIR, "train_images") if os.path.isdir(d)][0])
fp = REPO_DIR / "splits" / "folds.csv"
folds = pd.read_csv(fp) if fp.exists() else make_folds(ann.meta, cfg["data"]["n_folds"], SEED)
folds = folds[folds.stem.isin(ann.meta.stem)].reset_index(drop=True)
TR, VA = folds.stem[folds.fold != FOLD].tolist(), folds.stem[folds.fold == FOLD].tolist()
build_cache(ann, folds.stem.tolist(), CACHE_DIR, cfg, workers=WORKERS, log=log)
log(f"fold {FOLD}: {len(TR)} train / {len(VA)} validation images")

In [ ]:
# --------------------------------------------------------------------------------- teacher targets on GONG bursts
teacher = load_model(TEACHER, cfg, device)
tp.build_targets(GONG_DIR, RAW_DIR, teacher, TARGET_DIR, cfg, device, workers=WORKERS, log=log)
TARGET_TAR = globals().get("TARGET_TAR")                     # optional backup of the targets (e.g. on Drive)
if TARGET_TAR and not Path(TARGET_TAR).exists() and any((TARGET_DIR / "hedef").glob("*.png")):
    import subprocess
    subprocess.run(["tar", "-cf", str(TARGET_TAR), "-C", str(TARGET_DIR), "."], check=True)
    log(f"targets backed up -> {TARGET_TAR}")
del teacher
if device.type == "cuda": torch.cuda.empty_cache()
BURSTS = tp.student_bursts(GONG_DIR, TARGET_DIR)
assert BURSTS, ("no training burst has a target and two student frames: check the raw GONG frames (RAW_DIR) "
                f"and the targets in {TARGET_DIR}")
iou = []
for b, frames in list(BURSTS.items())[:200]:
    T = read_cache(TARGET_DIR, "hedef", b) > 0.5
    for s in frames:
        P = read_cache(TARGET_DIR, "tek", s) > 0.5
        iou.append((P & T).sum() / max((P | T).sum(), 1))
log(f"{len(BURSTS)} training bursts, {sum(len(v) for v in BURSTS.values())} student frames | "
    f"single-frame vs burst target IoU {np.mean(iou):.3f} (earlier ResNet teacher: 0.812)")

In [ ]:
# ------------------------------------------------------------------------------------------------- students
deadline = time.time() + TIME_BUDGET_H * 3600 if TIME_BUDGET_H else None
steps = len(TR) * cfg["data"]["crops_per_image"] // cfg["train"]["batch_size"]
STATUS = {}
for name in RUN:
    v = VARIANTS[name]
    aux = tp.make_aux(BURSTS, TARGET_DIR, v["target"], v["consistency"], cfg, steps,
                      ramp=(0, 1) if DRY else (2, 7), workers=cfg["train"]["workers"])
    r = train_model(cfg, f"cnx_{name}_fold{FOLD}", TR, VA, CACHE_DIR, WORK_DIR, device, seed=SEED + FOLD,
                    pretrained=not DRY, deadline=deadline, log=log, aux=aux)
    STATUS[name] = r["status"]
    if r["status"] != "done":
        break
ALL_DONE = all((WORK_DIR / "ckpt" / f"cnx_{v}_fold{FOLD}.done.json").exists() for v in VARIANTS)
log(f"status {STATUS} | all students trained: {ALL_DONE}")
if not ALL_DONE:
    raise SystemExit("Paused: run the notebook again to continue training.")

In [ ]:
# --------------------------------------------------------------------------- accuracy: fold-0 PQ (paired)
MODELS = {"baseline (final fold 0)": TEACHER, **{n: WORK_DIR / "ckpt" / f"cnx_{n}_fold{FOLD}.pt" for n in VARIANTS}}
groups = {s: i % 4 for i, s in enumerate(VA)}
R, rows = {}, []
for name, ck in MODELS.items():
    prob = pl.oof_probabilities(cfg, VA, {s: FOLD for s in VA}, {FOLD: ck}, CACHE_DIR,
                                WORK_DIR / f"prob_{name.split()[0]}", device, log=log)
    C = pl.candidates(prob, VA, ann, CACHE_DIR, cfg, workers=WORKERS, log=log)
    R[name] = pl.evaluate_scorer(C, VA, groups, cfg, SEED)
    s_all, s_sc = ev.summary(R[name]["P_all"]), ev.summary(R[name]["P"])
    rows.append(dict(model=name, pq=s_sc["PQ"], pq_all=s_all["PQ"], sq=s_sc["SQ"], rq=s_sc["RQ"], tp=s_sc["TP"],
                     fp=s_sc["FP"], fn=s_sc["FN"], tau=R[name]["tau"]))
ACC = pd.DataFrame(rows)
base = "baseline (final fold 0)"
cmp_ = []
for n in VARIANTS:
    b, ba = ev.paired_bootstrap(R[n]["P"], R[base]["P"]), ev.paired_bootstrap(R[n]["P_all"], R[base]["P_all"])
    cmp_.append(dict(comparison=f"{n} - baseline", d_pq=b["diff"], ci95=f"[{b['lo']:+.4f}, {b['hi']:+.4f}]",
                     d_pq_all=ba["diff"], ci95_all=f"[{ba['lo']:+.4f}, {ba['hi']:+.4f}]"))
ACC_CMP = pd.DataFrame(cmp_)
log("\n" + ACC.round(4).to_string(index=False) + "\n\n" + ACC_CMP.round(4).to_string(index=False))

In [ ]:
# ------------------------------------------------------------------- stability: self-PQ on held-out bursts
nets = {"baseline": load_model(TEACHER, cfg, device),
        **{n: load_model(WORK_DIR / "ckpt" / f"cnx_{n}_fold{FOLD}.pt", cfg, device) for n in VARIANTS}}
S = tp.stability(nets, GONG_DIR, cfg, device, done_path=WORK_DIR / "stability_per_burst.json", log=log)
del nets
rows = []
for n, r in S.items():
    rows.append(dict(model=n, self_pq_clean=ev.pq(r["tt"]), self_pq_with_cloudy=ev.pq(r["bi"]), pixel_iou=r["pixel_iou"],
                     count_std=r["count_std"], count_mean=r["count_mean"]))
STAB = pd.DataFrame(rows)
cmp_ = []
for a, b in (("demet_tc", "baseline"), ("tekkare", "baseline"), ("demet_tc", "tekkare")):
    for k, lab in (("tt", "clean"), ("bi", "with cloudy")):
        x = ev.paired_bootstrap(S[a][k], S[b][k])
        cmp_.append(dict(comparison=f"{a} - {b}", frames=lab, d_self_pq=x["diff"], ci95=f"[{x['lo']:+.4f}, {x['hi']:+.4f}]"))
STAB_CMP = pd.DataFrame(cmp_)
log("\n" + STAB.round(4).to_string(index=False) + "\n\n" + STAB_CMP.round(4).to_string(index=False))
log("Earlier ResNet reference: teacher 0.634 / 0.514, demet_tc 0.665 / 0.546, tekkare 0.617 / 0.499")
for nm, df in (("accuracy", ACC), ("accuracy_comparisons", ACC_CMP), ("stability", STAB), ("stability_comparisons", STAB_CMP)):
    df.to_csv(WORK_DIR / f"temporal_{nm}.csv", index=False)
log(f"saved -> {WORK_DIR}")